# Домашнее задание 2 (5 баллов).

*Все задания ниже имеют равный вес (5/10)*

Код для импорта мы написали за вас (не благодарите, нам не трудно). Дальше код будете писать вы.

[Тут](https://habr.com/ru/companies/ruvds/articles/494720/) шпора по pandas. За основу домашнего задания взят ноутбук [отсюда](https://rutube.ru/video/f884aa6ed5f94120b7304506042fe5bb/) (не подглядывайте!).

In [2]:
import pandas as pd
import numpy as np

#### Описание данных

Автор д/з - плохой человек, который не стал переводить описание с мотивировкой, что весь DS на английском. Так что описание полей будет на английском:

1. Account ID
- Description: A unique identifier for each social media account in the dataset.
- Type: Integer
- Example: 1, 2, 3, …
2. Username
- Description: The username or handle of the social media account.
- Type: String
- Example: john_doe, tech_guru_22, fitness_freak
3. Platform
- Description: The social media platform the account is using (Instagram, Twitter, Facebook, TikTok, LinkedIn).
- Type: Categorical (String)
- Example: Instagram, Twitter, Facebook, TikTok, LinkedIn
4. Follower Count
- Description: The total number of followers the account has.
- Type: Integer
- Example: 1500, 245000, 78000
5. Posts Per Week
- Description: The average number of posts the account creates per week.
- Type: Integer
- Example: 3, 5, 7
6. Engagement Rate
- Description: The percentage of interactions (likes, comments, shares) relative to the follower count. This is a measure of how engaging the content is.
- Type: Float
- Range: 0.01 to 0.15
- Example: 0.045 (4.5% engagement rate)
7. Ad Spend (USD)
- Description: The monthly amount spent on advertising or promoting posts.
- Type: Float
- Example: 150.75, 850.00, 300.50
8. Conversion Rate
- Description: The percentage of users who take a desired action (e.g., clicking a link, signing up, etc.) after interacting with an ad.
- Type: Float
- Range: 0.01 to 0.05 (1% to 5% conversion rate)
- Example: 0.025 (2.5% conversion rate)
9. Campaign Reach
- Description: The total number of unique users reached by the user’s campaigns in a given month.
- Type: Integer
- Example: 5000, 20000, 15000

#### Задание 0

Подгрузите данные. Да-да, за чтение таблицы баллов не будет))

**Hint**: [pd.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [3]:
!wget https://raw.githubusercontent.com/hse-ds/iad-intro-ds/refs/heads/master/2025/homeworks/hw02-pandas/data.csv

--2025-02-13 16:37:22--  https://raw.githubusercontent.com/hse-ds/iad-intro-ds/refs/heads/master/2025/homeworks/hw02-pandas/data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 595823 (582K) [text/plain]
Saving to: ‘data.csv’

data.csv            100%[===================>] 581.86K  3.12MB/s    in 0.2s    

2025-02-13 16:37:22 (3.12 MB/s) - ‘data.csv’ saved [595823/595823]



In [4]:
df = pd.read_csv("data.csv")

#### Задание 1

Колонка `Platform` содержит название различных платформ. Давайте представим, что в них есть некоторое отношение порядка. Закодируйте каждую платформу целым числом (от 0 до N) и положите этот "код" в новую колонку `Platform_Code`. Теперь вычислите корреляцию Спирмена между всеми парами колонок в датасете (результатом будет таблица корреляций). В качестве ответа выведите значение корреляции `Platform_Code` с `Engagement Rate`. Можете после вывода числа еще коротко написать, что оно означает (нет, это не оценивается).

**Hint**: [pd.factorize](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.factorize.html), [pd.DataFrame.select_dtypes](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html), [pd.DataFrame.corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html).

In [16]:
codes, uniques = pd.factorize(df["Platform"])
df["Platform_Code"] = codes
df.corr(method = "spearman",numeric_only=True)
print("Корреляция Спирмена для Platform_Code и Engagement Rate равна", df["Platform_Code"].corr(df["Engagement Rate"], method = "spearman"))

Корреляция Спирмена для Platform_Code и Engagement Rate равна 0.03138169529349812


#### Задание 2

Теперь посмотрите на столбец `Follower Count`. В нем какие-то числа. Иногда бывает полезно провести дискретизацию такого признака. Разбейте все значения в столбце на 4 группы: "Low", "Medium", "High", "Very High". Каждая группа включает в себя новые 25% данных. То есть, Low включает в себя 25% самых маленьких значений признака и так далее. Положите значения "Low", "Medium", "High" или "Very High" для каждого сэмпла датасета в новую колонку `Follower_Bin`. Теперь посчитайте среднее значение `Engagement Rate` для каждой категории из `Follower_Bin`. В качестве ответа выведите значение для категории "High".

**Hint**: [pd.qcut](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.qcut.html), [pd.groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html), [pd.DataFrame.mean](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html)

In [15]:
df["Follower_Bin"] = pd.qcut(df["Follower Count"], q=4, labels = ["Low", "Medium", "High", "Very High"])
average_engagement_rate = df.groupby(["Follower_Bin"], observed=False)["Engagement Rate"].mean()
print("Среднее значение Engagement Rate для категории High равно", average_engagement_rate.loc["High"])

Среднее значение Engagement Rate для категории High равно 0.08655032


#### Задание 3

Иногда бывает полезно превратить широкую таблицу в длинную (например, для визуализаций сразу нескольких признаков на одной картинке). Да, звучит странно, но именно этим вы сейчас и займетесь. Сделайте новый датафрейм `melted_df`, в который вы поместите каждый сэмпл датасета 6 раз: по одному разу на значение из 'Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate' и 'Campaign Reach'. То есть, вы берете сэмпл из датасета (строку) и превращаете ее в 6 отдельных строк. Каждая отдельная строка в столбце `Metric` имеет имя из предложенного списка 5 признаков, а в столбце `Value` - значение данного сэмпла по этому признаку. Значение `Platform` повторяется в этих 6 строках.

Иначе говоря,

```json
{
    "Account ID": 1,
    "Username": "harrislisa",
    "Platform": "TikTok",
    "Follower Count": 54217,
    "Posts Per Week": 3,
    "Engagement Rate": 0.0986,
    "Ad Spend (USD)": 538.1,
    "Conversion Rate": 0.049,
    "Campaign Reach": 1308,
    "Platform_Code": 0,
    "Follower_Bin": "Low"
}
```

превращается в

```json
{
    "Platform": "TikTok",
    "Metric": "Follower Count",
    "Value": 54217,
},
{
    "Platform": "TikTok",
    "Metric": "Posts Per Week",
    "Value": 3,
}, ...
```

Для каждого уникальной пары значений (`Platform`, `Metric`) посчитайте моду среди всех значений `Value` для этой пары, результат сделайте списком и оставьте только наибольшее. В качестве ответа выведите сумму полученных мод (сумму всех значений в столбце `Value` уже после вычисления мод). Иначе говоря, выведите сумму всех мод значений для всех уникальных пар (`Platform`, `Metric`).

**Hint**: [pd.melt](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html), [pd.DataFrame.mode](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mode.html), [pd.DataFrameGroupBy.agg](https://pandas.pydata.org/docs/dev/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

In [14]:
melted_df = pd.melt(df, id_vars=["Platform"], value_vars=['Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate', 'Campaign Reach'], var_name="Metric")
modes = melted_df.groupby(["Platform", "Metric"])["value"].agg(lambda x: x.mode().max())
print("Сумма всех мод значений для всех уникальных пар (Platform, Metric) равна", modes.sum())

Сумма всех мод значений для всех уникальных пар (Platform, Metric) равна 3100285.4716


#### Задание 4

А теперь хочется посмотреть на самые популярные аккаунты на разных платформах. Для каждой платформы отсортируйте датафрейм по убыванию количества подписчиков (`Follower Count`) - да, без циклов, сразу для всех платформ сделать сортировку, а затем оставьте только первые три записи для каждой платформы - это и будут три самых популярных аккаунта для каждой платформы. В качестве ответа выведите саму таблицу и минимальное значение `Follower Count` в ней.

**Hint**: к *groupby* можно применять функции - это эквивалентно применению функции к каждой "группе" внутри groupby-объекта. Читайте [про применение apply к датафрейму после groupby](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#flexible-apply).

In [13]:
groups = df.groupby(["Platform"]).apply(lambda x: x.sort_values("Follower Count", ascending=False).head(3))
groups

<ipython-input-13-46d77cfd22d5>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  groups = df.groupby(["Platform"]).apply(lambda x: x.sort_values("Follower Count", ascending=False).head(3))


Account ID         Username   Platform  Follower Count  \
Platform                                                                 
Facebook  2403        2404           eric65   Facebook          999982   
          7350        7351     patricknoble   Facebook          997915   
          1689        1690      chavezjason   Facebook          997512   
Instagram 8685        8686  alexandersamuel  Instagram          999726   
          3965        3966         lrodgers  Instagram          999351   
          2189        2190           jbrown  Instagram          997844   
LinkedIn  3039        3040          toneill   LinkedIn          999055   
          6359        6360    andrewgregory   LinkedIn          998968   
          2159        2160     ashleycooper   LinkedIn          998925   
TikTok    5838        5839     edwardthomas     TikTok          999739   
          4234        4235    andradewesley     TikTok          999234   
          2575        2576     williamwyatt     TikTok          998623   
Twitter   4920        4921      teresaellis    Twitter          999919   
          9684        9685           sriley    Twitter          999442   
          7576        7577       peggymunoz    Twitter          998216   

                Posts Per Week  Engagement Rate  Ad Spend (USD)  \
Platform                                                          
Facebook  2403               6           0.0642          884.06   
          7350               3           0.0834          429.01   
          1689               7           0.0834          993.20   
Instagram 8685               3           0.0834          687.61   
          3965               1           0.0834          565.07   
          2189               5           0.0642          505.61   
LinkedIn  3039               4           0.0642          799.49   
          6359               7           0.1020          797.64   
          2159               6           0.0856          474.46   
TikTok    5838               7           0.0642          630.77   
          4234               5           0.0834          872.77   
          2575               6           0.0856          477.98   
Twitter   4920               6           0.0834          411.63   
          9684               3           0.0834          206.84   
          7576               6           0.0642          456.61   

                Conversion Rate  Campaign Reach Conversion_Category  
Platform                                                             
Facebook  2403           0.0281           17312                 Low  
          7350           0.0182           25985                 Low  
          1689           0.0397           45717                High  
Instagram 8685           0.0205           11050                 Low  
          3965           0.0335           12391                High  
          2189           0.0202           14717                 Low  
LinkedIn  3039           0.0174           21862                 Low  
          6359           0.0351           15552                High  
          2159           0.0156           45956                 Low  
TikTok    5838           0.0325           35523                High  
          4234           0.0481           17188                High  
          2575           0.0250           43299                 Low  
Twitter   4920           0.0460            3975                High  
          9684           0.0225           12783                 Low  
          7576           0.0456           22037                High

In [27]:
print("Минимальное значение Follower Count в таблице groups равно", groups["Follower Count"].min())

Минимальное значение Follower Count в таблице groups равно 997512


#### Задание 5

Хочется посчитать какую-то метрику. Мы хотим посмотреть, на отношение разности суммы подписчиков аккаунтов с высокой и низкой конверсией к суммарному охвату рекламы на каждой платформе. То есть, мы делим аккаунты на две группы: высокая и низка конверсия. Затем мы смотрим на то, на сколько сильно влияние аккаунтов с высокой конверсией по сравнению с аккаунтами с низкой конверсией.

Давайте определим *Conversion Influence* следущим образом:

$$Conversion Influence = \frac{Total Follower\ Count (High) - Total Follower\ Count (Low)}{Total Campaign Reach (High)+Total Campaign Reach (Low)}$$

Считать эту метрику мы будет для каждой `Platform`. В этой формуле High - это значения всех сэмплов датасета, в которых `Conversion Rate` больше медианы, а `Low` - не более медианы. `Total Feature` - это суммарное количество значений `Feature` либо по `High` сэмплам, либо по `Low`.

Чтобы постоянно не пересчитывать, где High. где Low, сделайте новую колонку в датасете `Conversion_Category`. Положите в нее для каждой строки либо High, либо Low.

Выведите платформу с самым большим `Conversion Influence`.

**Hint**: данное задание не про *groupby*, а скорее про [pd.pivot_table](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html). Сделайте сводную таблицу, по которой уже можно посчитать суммы, а затем подставить их в формулы.

In [12]:
mediana = df["Conversion Rate"].median()
df["Conversion_Category"] = "High"
df.loc[df["Conversion Rate"] <= mediana, "Conversion_Category"] = "Low"
table = pd.pivot_table(df, values=["Follower Count", "Campaign Reach"], index=["Platform"], columns=["Conversion_Category"], aggfunc="count")
table["ConversionInfluence"] = (table[('Follower Count', 'High')] - table[('Follower Count',  'Low')]).abs() /(table[('Campaign Reach', 'High')] + table[('Campaign Reach',  'Low')])
index = table["ConversionInfluence"].idxmax()
platform = table.loc[index].name
print("Платформа с самым большим Conversion Influence -", platform)

Платформа с самым большим Conversion Influence - Twitter


#### Задание 6

Мы знаем, что вам понравилось считать метрики по формуле. Давайте закрепим этот успех. Теперь для каждой платформы посчитаем, на сколько эффективна реклама в разрезе трех последовательных записей в датасете.

Для каждой платформы отсортируйте записи в порядке убывания `Posts Per Week`. Будто бы аккаунты, которые постят чаще, используют более "активные" стратегии по рекламе. Теперь посчитайте *скользущие суммы с окном 3* по `Campaign Reach` и `Ad Spend (USD)`. Скользящая сумма с окном N - это вы идете по массиву, берете все последовательные тройки записей и суммируете их. Для первых двух записей троек не найдется. Для них скользящее среднее - NaN, что нам не помешает.

Теперь для каждого окна посчитайте

$$Rolling Efficiency Ratio = \frac{Rolling Sum of Campaign Reach}{Rolling Sum of Ad Spend}$$

По сути, для каждого окна вы посчитаете сколько пользователе привлеклось за один доллар, потреченный на рекламу, в данном окне. Понятно, что значений будет столько, сколько окон. Нам интересно максимально значение такой эффективности для каждой платформы.

В качестве ответа выведите название платформы с наибольшей максимальной эффективность и наименьшей (два названия, не одно, не три, ровно два).

**Hint**: окна можно делать через [pd.DataFrame.rolling](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html).

In [11]:
sort_groups = df.groupby(["Platform"]).apply(lambda x: x.sort_values("Posts Per Week", ascending=False))
rolsum_campaign_reach = df["Campaign Reach"].rolling(3).sum()
rolsum_ad_spend = df["Ad Spend (USD)"].rolling(3).sum()
rolling_Efficiency_Ratio = rolsum_campaign_reach / rolsum_ad_spend
index_min = rolling_Efficiency_Ratio.idxmin()
min_platform = sort_groups["Platform"].iloc[index_min]
index_max = rolling_Efficiency_Ratio.idxmax()
max_platform = sort_groups["Platform"].iloc[index_max]
print("Платформа с наибольшей максимальной эффективностью -", max_platform)
print("Платформа с наименьшей максимальной эффективностью -", min_platform)

Платформа с наибольшей максимальной эффективностью - LinkedIn
Платформа с наименьшей максимальной эффективностью - Facebook


<ipython-input-11-9c1aa37a83ce>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sort_groups = df.groupby(["Platform"]).apply(lambda x: x.sort_values("Posts Per Week", ascending=False))


#### Задание 7

Это еще не все прекрасные функции pandas, которые мы хотим вам показать. Теперь вы посчитаете, сколько аккаунтов на каждой платформе одновременно лучшие по `Engagement Rate` и `Conversion Rate`.

Сделайте два отдельных суб-сета. В одном оставьте для каждой платфмормы один топовый аккаунт по `Engagement Rate`, в другом - по `Conversion Rate`. Соедините эти два подмножества по столбцу `Platform` так, что в одно строке есть описание сразу двух аккаунтов-лидеров. Теперь посмотрите равны ли имена аккаунтов в одной строке. Выведите количество строк, в которых названия аккаунтов совпадают.

In [10]:
top_engagement_rate = df.groupby(["Platform"])["Engagement Rate"].idxmax()
top_engagement_rate_accounts = df.loc[top_engagement_rate]
top_conversion_rate = df.groupby(["Platform"])["Conversion Rate"].idxmax()
top_conversion_rate_accounts = df.loc[top_conversion_rate]
df_merged = pd.merge(top_engagement_rate_accounts, top_conversion_rate_accounts, on=['Platform'], how='inner')
result = (df_merged["Username_x"] == df_merged["Username_y"]).sum()
print("Количество строк, в которых названия аккаунтов совпадают -", result)

Количество строк, в которых названия аккаунтов совпадают - 0


#### Задание 8

Давайте теперь что-то попроще сделаем. Например, посчитаем отношение суммарного количества подписчиков на аккаунтах с высокой конверсией к такой же сумме в аккаунтах с низкой конверсией (очевидно, для каждой платформы). По сути, мы просто хотим получить число, которое характеризует, на сколько сильно аккаунты с высокой конверсией "доминируют" над аккаунтами с низкой конверсией в плане количества подписчиков.

Высокой конверсией будем считать конверсию больше средней. Остальное - низкая. Посчитайте суммы подписчиков для каждой платформы, поделите одно на другое и выведите разницу между самым большим значением и самым маленьким, а также платформы, которые соотвутствуют этим значениям.

Используйте магическую команду `%%time`, чтобы замерить, сколько времени ушло на исполнение вашего pandas-скрипта.

In [26]:
%%time
mean = df["Conversion Rate"].mean()
df["Conversion_Category"] = "High"
df.loc[df["Conversion Rate"] <= mean, "Conversion_Category"] = "Low"
table = pd.pivot_table(df, values=["Follower Count"], index=["Platform"], columns=["Conversion_Category"], aggfunc="sum")
table["Ratio"] = table[('Follower Count', 'High')] / table[('Follower Count',  'Low')]
min_platform = table['Ratio'].idxmin()
max_platform = table['Ratio'].idxmax()
raz = table["Ratio"].max() - table["Ratio"].min()
print("Платформа " + min_platform + " имеет максимальное соотношение сумм")
print("Платформа " + max_platform + " имеет минимальное соотношение сумм")
print("Разница между максимальным и минимальным значением:", raz)

Платформа Instagram имеет максимальное соотношение сумм
Платформа Twitter имеет минимальное соотношение сумм
Разница между максимальным и минимальным значением: 0.17688741338715763
CPU times: user 13.9 ms, sys: 872 µs, total: 14.8 ms
Wall time: 14.6 ms


#### Задание 9

А теперь решите задание 8 чисто питоном. Никаких функций и методов pandas. Только питоновские циклы. Замерьте время выполнения кода. Наконец, сравните время в задании 8 и 9. Напишите ниже, кто же победил: чистый python и pandas?

**Hint**: Чтобы итерироваться по датафрейму, можно из него сделать генератор через [pd.DataFrame.iterrows](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) или [pd.DataFrame.itertuples](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.itertuples.html#pandas.DataFrame.itertuples). К слову, это не все способы итерироваться по датафрейму.

In [25]:
%%time
d = dict()
for index, row in df.iterrows():
  if row["Platform"] not in d:
    d[row["Platform"]] = [0, 0]
  if row["Conversion_Category"] == "Low":
    d[row["Platform"]][0] += row["Follower Count"]
  else:
    d[row["Platform"]][1] += row["Follower Count"]
new_dict = dict()
for i in d:
  new_dict[i] = d[i][1] / d[i][0]
mx = -10**9
mn = 10**9
min_platform = ""
max_platform = ""
for i in new_dict:
  if new_dict[i] > mx:
    mx = new_dict[i]
    max_platform = i
  if new_dict[i] < mn:
    mn = new_dict[i]
    min_platform = i
print("Платформа " + max_platform + " имеет максимальное соотношение сумм")
print("Платформа " + min_platform + " имеет минимальное соотношение сумм")
print("Разница между максимальным и минимальным значением:", mx - mn)

Платформа Twitter имеет максимальное соотношение сумм
Платформа Instagram имеет минимальное соотношение сумм
Разница между максимальным и минимальным значением: 0.17688741338715763
CPU times: user 464 ms, sys: 1.08 ms, total: 465 ms
Wall time: 466 ms


**А победителем является**: pandas.
Код с использованием pandas работает в разы быстрее (14.6 ms), чем код на обычном питоне (466 ms). Это связано с тем, что pandas позволяет выполнять операции сразу над большим объемом данных (например, над столбцом). Это значительно ускоряет процесс выполнения программы, в отличие от питоновских циклов, где необходимо проходиться по всем элементам последовательно.

#### Задание 10

Крайне серьезное задание. Отнеситесь к нему соответствующе. В ячейке ниже напишите ваш любимый анекдот или мем (только без баянов, окей?). Можно плохие. Помните, это задание на полный балл. Проверяющий работу ассистент должен улыбнуться.

Если вставляете картинку, то убедитесь, что вы ее не подгружаете локально. А то будет неудобно - потерять балл на этом задании, когда надо было выложить картинку на облако и прокинуть ссылку. И нет, нельзя сюда просто ссылку вставить. Либо ищите, как вставить картинку, либо смешной анекдот. Есть всего два стула - выбирайте...

**Анекдот первый:**


-Продайте мне собаку


-Суку?


-Не, суку не надо, давайте нормальную


-Сука - это пол собаки


-Да зачем мне пол собаки? Давайте целую


-Зачем вы меня целуете


**Анекдот второй:**


Слепой заходит в магазин с собакой поводырем и начинает раскручивать собаку над головой


-Что вы делаете?


-Да так, осматриваюсь


**Парочка каламбуров:**


Как называется человек без левой ноги, без левой руки, без левого уха и без левого глаза?


-Allright


Как называется место на кладбище, где сидит охранник?


-Живой уголок


Как называют карлика в военкомате


-Маленький негодник


Что делают кофейные зерна перед смертью


-Молятся


Почему людоеды не едят бабушек?


-Они во рту вяжут


Почему нельзя драться с беременными?


-Они всегда готовы к схваткам


Куда положили тело су-шефа после смерти?


-В су-гроб